In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

kostasdiamantaras_chest_xrays_bacterial_viral_pneumonia_normal_path = kagglehub.dataset_download('kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal')
ibrahimfateen_wound_classification_path = kagglehub.dataset_download('ibrahimfateen/wound-classification')
pacificrm_skindiseasedataset_path = kagglehub.dataset_download('pacificrm/skindiseasedataset')

print('Data source import complete.')


100%|██████████| 1.14G/1.14G [00:21<00:00, 57.3MB/s]

Extracting files...


100%|██████████| 89.8M/89.8M [00:00<00:00, 144MB/s]

Extracting files...


100%|██████████| 1.36G/1.36G [00:14<00:00, 104MB/s]

Extracting files...


Data source import complete.


In [ ]:

# Optimized multi-backbone hybrid for 90%+ acc with sampler, mixup, early stopping, and backbone unfreeze

import os, torch, pandas as pd, numpy as np
from PIL import Image
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score
from torch.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ----------------- PATHS -----------------
CHEST_ROOT = "/root/.cache/kagglehub/datasets/kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal/versions/1"
SKIN_ROOT  = "/root/.cache/kagglehub/datasets/pacificrm/skindiseasedataset/versions/6/SkinDisease/SkinDisease"
WOUND_ROOT = "/root/.cache/kagglehub/datasets/ibrahimfateen/wound-classification/versions/8/Wound_dataset copy"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------- TRANSFORMS -----------------
train_transform_chest = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3,1,1)),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_transform_chest = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3,1,1)),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_transform_skin_wound = A.Compose([
    A.Resize(224,224),
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=(0.05,0.05), scale=(0.9,1.05), rotate=(-15,15), p=0.6),
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.5),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(224,224),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

# ----------------- DATASETS -----------------
class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None, val_frac=0.15):
        self.transform = transform
        df = pd.read_csv(os.path.join(root_dir, "labels_train.csv"))
        img_dir = os.path.join(root_dir, "train_images/train_images")
        df = df[df["file_name"].apply(lambda f: os.path.isfile(os.path.join(img_dir,f)))].reset_index(drop=True)
        train_df, val_df = train_test_split(df, test_size=val_frac, stratify=df["class_id"])
        self.df = train_df if split=="train" else val_df
        self.img_dir = img_dir
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["file_name"])).convert("L")
        if self.transform: img = self.transform(img)
        return img, int(row["class_id"])

class SkinDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        folder = os.path.join(root_dir,"train" if split=="train" else "test")
        classes = sorted([d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder,d))])
        self.classes = classes
        self.data = [(os.path.join(folder,cls,f),i)
                     for i,cls in enumerate(classes)
                     for f in os.listdir(os.path.join(folder,cls))
                     if f.lower().endswith((".jpg",".jpeg",".png"))]
        self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        path,label = self.data[idx]
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform: img = self.transform(image=img)["image"]
        return img,label

class WoundDatasetOnDemand(Dataset):
    def __init__(self, root_dir, val_split=0.1, transform=None):
        self.transform = transform
        classes = sorted([c for c in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,c))])
        self.classes = classes
        all_items = [(os.path.join(root_dir,c,f),i)
                     for i,c in enumerate(classes)
                     for f in os.listdir(os.path.join(root_dir,c))
                     if f.lower().endswith((".jpg",".jpeg",".png"))]
        train_len = int(len(all_items)*(1-val_split))
        self.train = all_items[:train_len]
        self.val = all_items[train_len:]
    def get_train(self): return self.train
    def get_val(self): return self.val

class WoundWrapper(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        path,label = self.data[idx]
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform: img = self.transform(image=img)["image"]
        return img,label

# ----------------- DATALOADERS -----------------
batch_size = 32
num_workers = 2

chest_train_ds = ChestXRayDataset(CHEST_ROOT,"train",train_transform_chest)
chest_val_ds = ChestXRayDataset(CHEST_ROOT,"val",val_transform_chest)
chest_loader = DataLoader(chest_train_ds,batch_size=batch_size,shuffle=True,num_workers=num_workers,pin_memory=True)
chest_valid_loader = DataLoader(chest_val_ds,batch_size=batch_size,shuffle=False,num_workers=num_workers,pin_memory=True)

skin_train_ds = SkinDataset(SKIN_ROOT,"train",train_transform_skin_wound)
skin_val_ds = SkinDataset(SKIN_ROOT,"test",val_transform)
# Weighted sampler for imbalanced classes
skin_labels = [lbl for _,lbl in skin_train_ds.data]
class_weights = compute_class_weight("balanced", classes=np.unique(skin_labels), y=skin_labels)
sample_weights = [class_weights[lbl] for lbl in skin_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
skin_loader = DataLoader(skin_train_ds,batch_size=batch_size,sampler=sampler,num_workers=num_workers,pin_memory=True)
skin_valid_loader = DataLoader(skin_val_ds,batch_size=batch_size,shuffle=False,num_workers=num_workers,pin_memory=True)

wound_ds = WoundDatasetOnDemand(WOUND_ROOT)
wound_train_ds = WoundWrapper(wound_ds.get_train(),train_transform_skin_wound)
wound_val_ds = WoundWrapper(wound_ds.get_val(),val_transform)
wound_loader = DataLoader(wound_train_ds,batch_size=batch_size,shuffle=True,num_workers=num_workers,pin_memory=True)
wound_valid_loader = DataLoader(wound_val_ds,batch_size=batch_size,shuffle=False,num_workers=num_workers,pin_memory=True)

# ----------------- MODEL -----------------
class MultiHeadFusionHybrid(nn.Module):
    def __init__(self, chest_classes, skin_classes, wound_classes):
        super().__init__()
        self.chest_backbone = nn.Sequential(*list(models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).children())[:-1])
        self.skin_backbone = nn.Sequential(*list(models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1).children())[:-1])
        self.wound_backbone = nn.Sequential(*list(models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features), nn.AdaptiveAvgPool2d(1))

        def make_head(in_dim,out_dim):
            return nn.Sequential(
                nn.Linear(in_dim,512), nn.ReLU(inplace=True), nn.Dropout(0.4),
                nn.Linear(512,256), nn.ReLU(inplace=True), nn.Dropout(0.25),
                nn.Linear(256,128), nn.ReLU(inplace=True), nn.Dropout(0.2),
                nn.Linear(128,out_dim)
            )

        self.chest_head = make_head(2048,chest_classes)
        self.skin_head = make_head(1536,skin_classes)
        self.wound_head = make_head(1024,wound_classes)

    def forward(self,x,dataset_type):
        if dataset_type=="chest":
            feat = self.chest_backbone(x).view(x.size(0),-1)
            return self.chest_head(feat)
        elif dataset_type=="skin":
            feat = self.skin_backbone(x).view(x.size(0),-1)
            return self.skin_head(feat)
        elif dataset_type=="wound":
            feat = self.wound_backbone(x).view(x.size(0),-1)
            return self.wound_head(feat)
        else:
            raise ValueError("dataset_type must be 'chest','skin','wound'")

# ----------------- LOSS & OPTIMIZER -----------------
chest_classes = len(set([lbl for _,lbl in chest_train_ds.df["class_id"].items()]))
skin_classes = len(skin_train_ds.classes)
wound_classes = len(wound_ds.classes)

model = MultiHeadFusionHybrid(chest_classes,skin_classes,wound_classes).to(device)
# Freeze skin backbone initially
for p in model.skin_backbone.parameters(): p.requires_grad = False

# Weighted CE losses with label smoothing
chest_labels = [lbl for _,lbl in chest_train_ds.df["class_id"].items()]
criterion_chest = nn.CrossEntropyLoss(weight=torch.tensor(compute_class_weight("balanced", classes=np.unique(chest_labels), y=chest_labels),dtype=torch.float).to(device), label_smoothing=0.1)
criterion_skin = nn.CrossEntropyLoss(label_smoothing=0.1)
wound_labels = [lbl for _,lbl in wound_train_ds.data]
criterion_wound = nn.CrossEntropyLoss(weight=torch.tensor(compute_class_weight("balanced", classes=np.arange(wound_classes), y=wound_labels),dtype=torch.float).to(device), label_smoothing=0.1)

optimizer = optim.Adam([
    {"params": model.chest_backbone.parameters(),"lr":1e-4},
    {"params": model.skin_backbone.parameters(),"lr":5e-4},  # slightly higher
    {"params": model.wound_backbone.parameters(),"lr":1e-4},
    {"params": model.chest_head.parameters(),"lr":2e-3},
    {"params": model.skin_head.parameters(),"lr":5e-3},  # higher for skin head
    {"params": model.wound_head.parameters(),"lr":2e-3}
], weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=40)
scaler = GradScaler()

# ----------------- MIXUP -----------------
def mixup_data(x,y,alpha=0.3):
    lam = np.random.beta(alpha,alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam*x + (1-lam)*x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam
def mixup_criterion(criterion,pred,y_a,y_b,lam):
    return lam*criterion(pred,y_a)+(1-lam)*criterion(pred,y_b)

# ----------------- TRAIN / VALID FUNCTIONS -----------------
def train_epoch(model,loader,optimizer,criterion,dataset_type,mixup=False):
    model.train()
    running_loss,all_preds,all_labels = 0.0,[],[]
    for imgs,labels in loader:
        imgs,labels = imgs.to(device),labels.to(device)
        optimizer.zero_grad()
        with autocast(device_type="cuda"):
            if mixup:
                imgs,y_a,y_b,lam = mixup_data(imgs,labels)
                outputs = model(imgs,dataset_type)
                loss = mixup_criterion(criterion,outputs,y_a,y_b,lam)
            else:
                outputs = model(imgs,dataset_type)
                loss = criterion(outputs,labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        all_preds += outputs.argmax(1).cpu().tolist()
        all_labels += labels.cpu().tolist()
    return running_loss/len(loader), accuracy_score(all_labels,all_preds)

def valid_epoch(model,loader,criterion,dataset_type):
    model.eval()
    running_loss,all_preds,all_labels = 0.0,[],[]
    with torch.no_grad():
        for imgs,labels in loader:
            imgs,labels = imgs.to(device),labels.to(device)
            with autocast(device_type="cuda"):
                outputs = model(imgs,dataset_type)
                loss = criterion(outputs,labels)
            running_loss += loss.item()
            all_preds += outputs.argmax(1).cpu().tolist()
            all_labels += labels.cpu().tolist()
    return running_loss/len(loader), accuracy_score(all_labels,all_preds)

# ----------------- TRAIN LOOP WITH BACKBONE UNFREEZE & EARLY STOPPING -----------------
num_epochs = 50
best_skin_acc = 0.0
skin_backbone_unfrozen = False
checkpoint_path = "best_skin_model.pth"

for epoch in range(num_epochs):
    print(f"\n=== Epoch {epoch+1}/{num_epochs} ===")

    # --- Chest ---
    tl,ta = train_epoch(model,chest_loader,optimizer,criterion_chest,"chest")
    vl,va = valid_epoch(model,chest_valid_loader,criterion_chest,"chest")
    print(f"[Chest] Train Loss: {tl:.4f}, Train Acc: {ta:.4f} | Val Acc: {va:.4f}")

    # --- Skin ---
    if not skin_backbone_unfrozen and epoch >= 5:
        for p in model.skin_backbone.parameters(): p.requires_grad = True
        print("*** Skin backbone unfrozen for fine-tuning ***")
        skin_backbone_unfrozen = True

    tl,ta = train_epoch(model,skin_loader,optimizer,criterion_skin,"skin", mixup=True)
    vl,va = valid_epoch(model,skin_valid_loader,criterion_skin,"skin")
    print(f"[Skin] Train Loss: {tl:.4f}, Train Acc: {ta:.4f} | Val Acc: {va:.4f}")

    if va > best_skin_acc:
        best_skin_acc = va
        torch.save(model.state_dict(), checkpoint_path)
        print(f"*** Best skin model saved with val_acc={best_skin_acc:.4f} ***")

    # --- Wound ---
    tl,ta = train_epoch(model,wound_loader,optimizer,criterion_wound,"wound", mixup=True)
    vl,va = valid_epoch(model,wound_valid_loader,criterion_wound,"wound")
    print(f"[Wound] Train Loss: {tl:.4f}, Train Acc: {ta:.4f} | Val Acc: {va:.4f}")

    scheduler.step()

print("\nTraining complete ✅")
print("Chest classes:",chest_classes,"Skin classes:",skin_classes,"Wound classes:",wound_classes)


Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 104MB/s] 


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 108MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 108MB/s]



=== Epoch 1/50 ===


/tmp/ipykernel_2671/2043107773.py:196: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_2671/2043107773.py:216: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type="cuda"):


KeyboardInterrupt: 

In [ ]:
torch.save(model, "hybrid_multitask_model.pth")
print("Hybrid model saved as hybrid_multitask_model.pth ✅")

In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------- LOAD SAVED MODEL -----------------
model = torch.load("hybrid_multitask_model.pth", map_location=device)
model.eval()

# ----------------- VALID TRANSFORM -----------------
val_transform = A.Compose([
    A.Resize(224,224),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

# ----------------- EVALUATION FUNCTION -----------------
def evaluate_model(model, dataset_type, dataset):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for img, label in dataset:
            # Apply transform if img is PIL image
            if not torch.is_tensor(img):
                img = val_transform(image=np.array(img))["image"]
            img = img.unsqueeze(0).to(device)  # Add batch dimension
            output = model(img, dataset_type)
            pred = output.argmax(1).item()
            all_preds.append(pred)
            all_labels.append(label)
    acc = accuracy_score(all_labels, all_preds)
    print(f"Accuracy on {dataset_type}: {acc:.4f}")
    print(classification_report(all_labels, all_preds))

# ----------------- TEST ON VALIDATION DATASETS -----------------
# Chest
chest_val_ds = ChestXRayDataset(CHEST_ROOT, split="val", transform=val_transform)
evaluate_model(model, "chest", chest_val_ds)

# Skin
skin_val_ds = SkinDataset(SKIN_ROOT, split="test", transform=val_transform)
evaluate_model(model, "skin", skin_val_ds)

# Wound
wound_ds = WoundDatasetOnDemand(WOUND_ROOT)
wound_val_ds = WoundWrapper(wound_ds.get_val(), transform=val_transform)
evaluate_model(model, "wound", wound_val_ds)

# New Section